# Comparison

> **Source:** `repo1/testing_patterns.py` → `run_comparison()`


## Imports


In [ ]:
import pytest
from unittest.mock import Mock, patch
from typing import Callable
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage
from langsmith import traceable, Client
from dotenv import load_dotenv
from langsmith import Client
from langsmith.evaluation import evaluate
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langsmith import traceable
from dotenv import load_dotenv


## Setup


In [ ]:
load_dotenv()

load_dotenv()

client = Client()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template("Answer this question concisely: {question}")

qa_chain = prompt | llm

eval_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


## Helper: `helpfulness`


In [ ]:
def helpfulness(run, example) -> dict:
    """LLM-as-judge evaluator for helpfulness (no reference needed)."""
    prediction = run.outputs.get("answer", "")
    question = example.inputs.get("question", "")

    grade_prompt = ChatPromptTemplate.from_template(
        "You are a grader. Given a question and a response, "
        "determine if the response is helpful, clear, and easy to understand.\n\n"
        "Question: {question}\n"
        "Response: {response}\n\n"
        "Respond with ONLY 'Y' if helpful or 'N' if not helpful."
    )
    result = eval_llm.invoke(
        grade_prompt.format(question=question, response=prediction)
    )
    score = 1.0 if result.content.strip().upper() == "Y" else 0.0
    return {"key": "helpfulness", "score": score}


## Helper: `correctness`


In [ ]:
def correctness(run, example) -> dict:
    """LLM-as-judge evaluator for correctness against reference answer."""
    prediction = run.outputs.get("answer", "")
    reference = example.outputs.get("answer", "")
    question = example.inputs.get("question", "")

    grade_prompt = ChatPromptTemplate.from_template(
        "You are a grader. Given a question, a submission, and a reference answer, "
        "determine if the submission is correct, accurate, and factual compared to "
        "the reference answer.\n\n"
        "Question: {question}\n"
        "Submission: {submission}\n"
        "Reference: {reference}\n\n"
        "Respond with ONLY 'Y' if correct or 'N' if incorrect."
    )
    result = eval_llm.invoke(
        grade_prompt.format(
            question=question, submission=prediction, reference=reference
        )
    )
    score = 1.0 if result.content.strip().upper() == "Y" else 0.0
    return {"key": "correctness", "score": score}


## Helper: `contains_answer`


In [ ]:
def contains_answer(run, example) -> dict:
    """
    Custom evaluator — checks if the response contains
    key terms from the expected answer.
    """
    prediction = run.outputs.get("answer", "").lower()
    reference = example.outputs.get("answer", "").lower()

    # Extract key words from reference (words > 3 chars)
    key_words = [word for word in reference.split() if len(word) > 3]

    # Check if at least 50% of key words appear in prediction
    if not key_words:
        return {"key": "contains_answer", "score": 1.0}

    matches = sum(1 for word in key_words if word in prediction)
    score = matches / len(key_words)

    return {"key": "contains_answer", "score": score}


## Demo: Comparison


In [ ]:
def run_comparison(dataset_name: str):
    """
    Run a second experiment with a different config,
    then compare in LangSmith dashboard.
    """

    # New prompt — more detailed instructions
    detailed_prompt = ChatPromptTemplate.from_template(
        "Answer this question accurately and concisely. "
        "If it's a factual question, be precise. "
        "If it's a math question, show just the answer.\n\n"
        "Question: {question}"
    )
    v2_chain = detailed_prompt | llm

    @traceable(name="qa_target_v2")
    def qa_target_v2(inputs: dict) -> dict:
        response = v2_chain.invoke({"question": inputs["question"]})
        return {"answer": response.content}

    print("\nRunning v2 experiment for comparison...\n")

    results = evaluate(
        qa_target_v2,
        data=dataset_name,
        evaluators=[correctness, helpfulness, contains_answer],
        experiment_prefix="qa-chain-v2",  # Different prefix for comparison
        max_concurrency=2,
    )

    print("\nDone! Compare v1 vs v2 in LangSmith dashboard:")
    print("  → Go to your LangSmith project → Datasets → qa-eval-dataset")
    print("  → Click 'Compare Experiments' to see v1 vs v2 side by side")

    return results


## Run


In [ ]:
run_comparison()
